# CLASIFICADOR DE IMÁGENES HÍBRIDO CLÁSICO/CUÁNTICO — v2 (TFM)

Versión robustecida del prototipo inicial. Cambios respecto a la v1:

- **Reproducibilidad**: semillas fijas en todas las fuentes de aleatoriedad.
- **Split train/val/test**: la v1 solo tenía train/test, sin validación (no se puede hacer early stopping ni detectar overfitting sin esto).
- **Configuración centralizada**: todos los hiperparámetros en un único objeto `Config`, para poder lanzar experimentos variando un parámetro a la vez.
- **Early stopping + checkpointing**: se guarda el mejor modelo según pérdida de validación, no el último.
- **Medición de tiempo de entrenamiento**: crítico en un TFM cuántico — el coste computacional es parte del resultado, no un detalle.
- **Baseline clásico equivalente**: mismo encoder y clasificador, pero sustituyendo la capa cuántica por una capa clásica del mismo tamaño. Esto aísla la contribución real de la parte cuántica (es el experimento más importante del TFM).
- **Baseline CNN de referencia**: para saber cuál es el techo de precisión realista en MNIST y contextualizar los resultados híbridos.
- **Métricas completas**: matriz de confusión, classification report, no solo accuracy global.
- **Logging estructurado**: histórico de cada experimento guardado en CSV para comparar entre configuraciones.

Tutorial original (con errores) que sirvió de punto de partida:
https://medium.com/@anirudhsekar2008/building-a-hybrid-quantum-classical-image-classifier-with-pytorch-and-pennylane-bb2ec5a1b030

## 0. Instalación de dependencias

In [ ]:
!pip install torch torchvision torchaudio pennylane scikit-learn matplotlib pandas -q

## 1. Imports, reproducibilidad y configuración

Todo lo que antes eran variables sueltas ahora vive en un único objeto `Config`. Esto es importante para un TFM porque vas a lanzar el mismo pipeline con distintos valores de `n_qubits`, `n_layers`, `entangler`, etc., y necesitas que cada corrida quede identificada y sea reproducible.

In [ ]:
import os
import json
import time
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import confusion_matrix, classification_report

import pennylane as qml


def set_seed(seed: int):
    """Fija la semilla en todas las fuentes de aleatoriedad relevantes.
    Sin esto, dos corridas con la misma config pueden dar accuracies distintas
    y no podrás saber si una mejora es real o es ruido del split/init."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class Config:
    # Identificación del experimento (para el CSV de resultados)
    run_name: str = "hybrid_baseline"

    # Reproducibilidad
    seed: int = 42

    # Arquitectura cuántica
    n_qubits: int = 4
    n_layers: int = 2
    entangler: str = "basic"        # "basic" | "strong"

    # Arquitectura clásica
    encoder_hidden_dim: int = 64

    # Entrenamiento
    batch_size: int = 64
    learning_rate: float = 1e-3
    epochs: int = 15
    val_split: float = 0.1          # % del train que se reserva para validación
    early_stopping_patience: int = 4

    # Infra
    device: str = "cpu"             # default.qubit de PennyLane funciona en CPU
    data_dir: str = "./data"
    checkpoint_dir: str = "./checkpoints"
    results_csv: str = "./results/experiments.csv"

    def as_dict(self):
        return asdict(self)


DEVICE = torch.device("cpu")
print(f"Dispositivo: {DEVICE}")
print("Nota: PennyLane default.qubit simula en CPU; usar GPU aquí no acelera la parte cuántica.")

## 2. Datos: carga con split train / val / test

**Bug de la v1**: solo había `trainset` y `testset`. Sin partición de validación no puedes hacer early stopping ni detectar overfitting durante el entrenamiento — el test set debe quedar intocado hasta la evaluación final, nunca usarse para tomar decisiones sobre el modelo.

In [ ]:
def get_dataloaders(cfg: Config):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    full_train = torchvision.datasets.MNIST(
        root=cfg.data_dir, train=True, download=True, transform=transform
    )
    test_set = torchvision.datasets.MNIST(
        root=cfg.data_dir, train=False, download=True, transform=transform
    )

    n_val = int(len(full_train) * cfg.val_split)
    n_train = len(full_train) - n_val

    # Generador con semilla propia para que el split sea reproducible
    generator = torch.Generator().manual_seed(cfg.seed)
    train_set, val_set = random_split(full_train, [n_train, n_val], generator=generator)

    train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=cfg.batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=cfg.batch_size, shuffle=False)

    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
    return train_loader, val_loader, test_loader

## 3. Circuito cuántico (parametrizable)

**Mejora sobre la v1**: el tipo de entrelazamiento ahora es un parámetro de configuración (`basic` vs `strong`), no algo hardcodeado. Así puedes comparar `BasicEntanglerLayers` contra `StronglyEntanglingLayers` como un experimento más, sin duplicar código.

In [ ]:
def build_quantum_layer(cfg: Config):
    dev = qml.device("default.qubit", wires=cfg.n_qubits)

    if cfg.entangler == "basic":
        weight_shape = (cfg.n_layers, cfg.n_qubits)

        @qml.qnode(dev, interface="torch")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(cfg.n_qubits))
            qml.BasicEntanglerLayers(weights, wires=range(cfg.n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(cfg.n_qubits)]

    elif cfg.entangler == "strong":
        weight_shape = (cfg.n_layers, cfg.n_qubits, 3)

        @qml.qnode(dev, interface="torch")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(cfg.n_qubits))
            qml.StronglyEntanglingLayers(weights, wires=range(cfg.n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(cfg.n_qubits)]

    else:
        raise ValueError(f"entangler desconocido: {cfg.entangler}")

    weight_shapes = {"weights": weight_shape}
    return qml.qnn.TorchLayer(circuit, weight_shapes)

## 4. Modelos

Tres modelos, pensados para que las comparaciones sean justas (misma arquitectura base):

1. **`HybridModel`** — encoder clásico → capa cuántica → clasificador. El modelo del TFM.
2. **`ClassicalEquivalentModel`** — idéntico, pero la capa cuántica se sustituye por una capa clásica (`Linear + Tanh`) del mismo tamaño. **Este es el experimento clave**: si el híbrido no supera a este baseline, la parte cuántica no está aportando nada medible.
3. **`SimpleCNNBaseline`** — una CNN convencional, para saber el techo de precisión alcanzable en MNIST y dar contexto a los resultados ("un modelo clásico simple llega a ~98%").

In [ ]:
class SharedEncoder(nn.Module):
    """Comprime 28x28 -> n_qubits valores en [-1, 1], compartido por el modelo híbrido
    y el baseline clásico equivalente para que la comparación sea justa."""
    def __init__(self, cfg: Config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, cfg.encoder_hidden_dim),
            nn.ReLU(),
            nn.Linear(cfg.encoder_hidden_dim, cfg.n_qubits),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.net(x)


class HybridModel(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.encoder = SharedEncoder(cfg)
        self.quantum_layer = build_quantum_layer(cfg)
        self.classifier = nn.Linear(cfg.n_qubits, 10)

    def forward(self, x):
        x = self.encoder(x) * torch.pi
        x = self.quantum_layer(x)
        return self.classifier(x)


class ClassicalEquivalentModel(nn.Module):
    """Mismo pipeline que HybridModel pero sin cuántica: capa Linear+Tanh del mismo
    tamaño en lugar del circuito. Es el ablation study más importante del TFM."""
    def __init__(self, cfg: Config):
        super().__init__()
        self.encoder = SharedEncoder(cfg)
        self.classical_layer = nn.Sequential(
            nn.Linear(cfg.n_qubits, cfg.n_qubits),
            nn.Tanh(),
        )
        self.classifier = nn.Linear(cfg.n_qubits, 10)

    def forward(self, x):
        x = self.encoder(x) * torch.pi
        x = self.classical_layer(x)
        return self.classifier(x)


class SimpleCNNBaseline(nn.Module):
    """CNN de referencia, sin relación con el pipeline híbrido. Sirve para contextualizar:
    ¿qué tan lejos está el modelo híbrido del techo de precisión en MNIST?"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)

## 5. Entrenamiento genérico: early stopping, checkpointing y medición de tiempo

**Bugs/carencias de la v1** que se corrigen aquí:
- Entrenaba un número fijo de épocas sin mirar validación → no había forma de saber si estaba sobreajustando ni de parar en el mejor punto.
- No se guardaba el modelo en ningún momento → si el kernel se caía, se perdía todo.
- No se medía el tiempo → para un TFM cuántico, el tiempo de entrenamiento es un resultado tan importante como la precisión (es el trade-off central del tema).

In [ ]:
def train_model(model, cfg: Config, train_loader, val_loader, run_name: str):
    Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    ckpt_path = Path(cfg.checkpoint_dir) / f"{run_name}.pt"

    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate)

    best_val_loss = float("inf")
    epochs_without_improvement = 0
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}

    start_time = time.time()

    for epoch in range(cfg.epochs):
        # --- train ---
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        # --- val ---
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc = correct / total

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"[{run_name}] Época {epoch+1}/{cfg.epochs} | "
              f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

        # --- checkpointing + early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= cfg.early_stopping_patience:
                print(f"[{run_name}] Early stopping en época {epoch+1} "
                      f"(sin mejora durante {cfg.early_stopping_patience} épocas)")
                break

    training_time = time.time() - start_time

    # Restaurar el mejor checkpoint antes de devolver el modelo
    model.load_state_dict(torch.load(ckpt_path))

    return model, pd.DataFrame(history), training_time

## 6. Evaluación completa (no solo accuracy)

La v1 solo reportaba un porcentaje de acierto global. Para el TFM necesitas poder argumentar *dónde* falla el modelo: si confunde sistemáticamente ciertos dígitos, eso es información relevante sobre las limitaciones del cuello de botella de 4 qubits.

In [ ]:
def evaluate_model(model, test_loader, run_name: str):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    print(f"\n[{run_name}] Accuracy en test: {acc*100:.2f}%\n")
    print(classification_report(all_labels, all_preds, digits=3))

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"Matriz de confusión — {run_name}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()

    return acc, cm

## 7. Registro de experimentos en CSV

Cada vez que se corre este notebook con una `Config` distinta, se añade una fila. Con esto, al final tendrás una tabla comparando todas las combinaciones de `n_qubits` / `n_layers` / `entangler` probadas — el insumo directo para la tabla de resultados de la memoria del TFM.

In [ ]:
def log_experiment(cfg: Config, model_type: str, test_acc: float, training_time: float, n_params: int):
    Path(cfg.results_csv).parent.mkdir(parents=True, exist_ok=True)
    row = {
        **cfg.as_dict(),
        "model_type": model_type,
        "test_accuracy": test_acc,
        "training_time_sec": round(training_time, 1),
        "n_trainable_params": n_params,
    }
    df_row = pd.DataFrame([row])
    if Path(cfg.results_csv).exists():
        df_row.to_csv(cfg.results_csv, mode="a", header=False, index=False)
    else:
        df_row.to_csv(cfg.results_csv, mode="w", header=True, index=False)
    print(f"Resultado guardado en {cfg.results_csv}")


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 8. Ejecución: modelo híbrido

Con `epochs=15` y early stopping en vez de los 3 fijos de la v1. Ajusta `n_qubits`, `n_layers` o `entangler` en el `Config` para lanzar variantes distintas — cada una quedará registrada por separado en el CSV de resultados.

In [ ]:
cfg = Config(run_name="hybrid_q4_l2_basic", n_qubits=4, n_layers=2, entangler="basic", epochs=15)
set_seed(cfg.seed)

train_loader, val_loader, test_loader = get_dataloaders(cfg)

hybrid_model = HybridModel(cfg)
hybrid_model, history_hybrid, time_hybrid = train_model(
    hybrid_model, cfg, train_loader, val_loader, run_name=cfg.run_name
)
acc_hybrid, cm_hybrid = evaluate_model(hybrid_model, test_loader, run_name=cfg.run_name)
log_experiment(cfg, "hybrid", acc_hybrid, time_hybrid, count_params(hybrid_model))

## 9. Ejecución: baseline clásico equivalente

**Este es el experimento que responde a la pregunta central del TFM**: ¿la capa cuántica aporta algo que una capa clásica del mismo tamaño no consiga? Se usa la misma `Config` (mismo seed, mismo split) para que la comparación sea limpia.

In [ ]:
classical_model = ClassicalEquivalentModel(cfg)
classical_model, history_classical, time_classical = train_model(
    classical_model, cfg, train_loader, val_loader, run_name="classical_equivalent"
)
acc_classical, cm_classical = evaluate_model(classical_model, test_loader, run_name="classical_equivalent")
log_experiment(cfg, "classical_equivalent", acc_classical, time_classical, count_params(classical_model))

print(f"\n--- Comparación directa ---")
print(f"Híbrido:              {acc_hybrid*100:.2f}%  ({time_hybrid:.1f}s)")
print(f"Clásico equivalente:  {acc_classical*100:.2f}%  ({time_classical:.1f}s)")

## 10. Ejecución: baseline CNN de referencia (opcional, para contexto)

In [ ]:
cnn_model = SimpleCNNBaseline()
cnn_model, history_cnn, time_cnn = train_model(cnn_model, cfg, train_loader, val_loader, run_name="cnn_reference")
acc_cnn, cm_cnn = evaluate_model(cnn_model, test_loader, run_name="cnn_reference")
log_experiment(cfg, "cnn_reference", acc_cnn, time_cnn, count_params(cnn_model))

## 11. Curvas de entrenamiento comparadas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_hybrid["epoch"], history_hybrid["val_acc"], label="Híbrido")
axes[0].plot(history_classical["epoch"], history_classical["val_acc"], label="Clásico equivalente")
axes[0].plot(history_cnn["epoch"], history_cnn["val_acc"], label="CNN referencia")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Val accuracy")
axes[0].set_title("Precisión de validación por época")
axes[0].legend()

axes[1].plot(history_hybrid["epoch"], history_hybrid["val_loss"], label="Híbrido")
axes[1].plot(history_classical["epoch"], history_classical["val_loss"], label="Clásico equivalente")
axes[1].plot(history_cnn["epoch"], history_cnn["val_loss"], label="CNN referencia")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Val loss")
axes[1].set_title("Pérdida de validación por época")
axes[1].legend()

plt.tight_layout()
plt.show()

## 12. Tabla acumulada de experimentos

Cada corrida de este notebook con una `Config` distinta añade filas aquí. Esta tabla es la base directa para la sección de resultados de la memoria.

In [ ]:
results_df = pd.read_csv(cfg.results_csv)
results_df[["run_name", "model_type", "n_qubits", "n_layers", "entangler",
            "test_accuracy", "training_time_sec", "n_trainable_params"]]

## 13. Barrido de hiperparámetros: n_qubits × n_layers

Aquí se recorre automáticamente la rejilla `n_qubits ∈ {4, 6, 8}` × `n_layers ∈ {2, 3, 4}` (9 combinaciones) reutilizando todo lo ya definido arriba (`Config`, `HybridModel`, `train_model`, `evaluate_model`, `log_experiment`). No hay que tocar nada de lo anterior, solo lanzar esta celda.

**Aviso de tiempo de cómputo**: cada combinación entrena un circuito cuántico simulado — en CPU esto es lento, y 9 combinaciones con `epochs=15` puede tardar varias horas. Por eso el barrido usa `epochs` más bajo y `patience` más corto (`sweep_epochs`, `sweep_patience` abajo) para explorar rápido qué zona de la rejilla es prometedora. Una vez identificada la mejor combinación, se reentrena esa configuración específica con el `epochs=15` completo de la Sección 8 para obtener el número final que va a la memoria.

In [ ]:
# Rejilla a explorar. Ajusta estas listas si quieres un barrido más fino o más amplio.
QUBITS_GRID = [4, 6, 8]
LAYERS_GRID = [2, 3, 4]

# Config reducida para explorar rápido (no es el número final, solo para comparar tendencias)
SWEEP_EPOCHS = 8
SWEEP_PATIENCE = 3

sweep_results = []

for nq in QUBITS_GRID:
    for nl in LAYERS_GRID:
        run_name = f"sweep_q{nq}_l{nl}_basic"
        print(f"\n=== Entrenando {run_name} ===")

        cfg_sweep = Config(
            run_name=run_name,
            n_qubits=nq,
            n_layers=nl,
            entangler="basic",
            epochs=SWEEP_EPOCHS,
            early_stopping_patience=SWEEP_PATIENCE,
        )
        set_seed(cfg_sweep.seed)

        # Mismo seed en todas las corridas => mismo split train/val/test,
        # así la única variable que cambia entre corridas es la arquitectura.
        tr_loader, va_loader, te_loader = get_dataloaders(cfg_sweep)

        model = HybridModel(cfg_sweep)
        model, history, elapsed = train_model(model, cfg_sweep, tr_loader, va_loader, run_name=run_name)
        acc, _ = evaluate_model(model, te_loader, run_name=run_name)
        log_experiment(cfg_sweep, "hybrid_sweep", acc, elapsed, count_params(model))

        sweep_results.append({
            "n_qubits": nq,
            "n_layers": nl,
            "test_accuracy": acc,
            "training_time_sec": round(elapsed, 1),
        })

sweep_df = pd.DataFrame(sweep_results)
sweep_df

### 13.1 Tabla y mapa de calor: accuracy por combinación

El pivote qubits × layers deja ver de un vistazo si la precisión sube más al añadir qubits (más ancho de información) o al añadir capas (más profundidad de entrelazamiento) — esa es una de las preguntas que puedes responder en la sección de resultados del TFM.

In [ ]:
pivot_acc = sweep_df.pivot(index="n_layers", columns="n_qubits", values="test_accuracy")
display(pivot_acc)

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(pivot_acc.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(pivot_acc.columns)))
ax.set_xticklabels(pivot_acc.columns)
ax.set_yticks(range(len(pivot_acc.index)))
ax.set_yticklabels(pivot_acc.index)
ax.set_xlabel("n_qubits")
ax.set_ylabel("n_layers")
ax.set_title("Accuracy en test según n_qubits / n_layers")

for i in range(pivot_acc.shape[0]):
    for j in range(pivot_acc.shape[1]):
        val = pivot_acc.values[i, j]
        ax.text(j, i, f"{val*100:.1f}%", ha="center", va="center", color="white")

fig.colorbar(im, label="Accuracy")
plt.tight_layout()
plt.show()

### 13.2 Accuracy vs tiempo de entrenamiento

Este gráfico es el que sustenta el argumento de coste-beneficio: si un punto está arriba a la izquierda (alta precisión, poco tiempo), esa combinación es más eficiente que una arriba a la derecha con la misma precisión pero mucho más lenta.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for nl in LAYERS_GRID:
    subset = sweep_df[sweep_df["n_layers"] == nl].sort_values("n_qubits")
    ax.plot(subset["training_time_sec"], subset["test_accuracy"], marker="o", label=f"n_layers={nl}")
    for _, row in subset.iterrows():
        ax.annotate(f"q{int(row.n_qubits)}", (row.training_time_sec, row.test_accuracy),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)

ax.set_xlabel("Tiempo de entrenamiento (s)")
ax.set_ylabel("Accuracy en test")
ax.set_title("Coste computacional vs precisión")
ax.legend()
plt.tight_layout()
plt.show()

### 13.3 Reentrenar la mejor combinación con el `epochs` completo

Una vez visto qué combinación de la rejilla rindió mejor en el barrido rápido, reentrénala con la configuración completa (15 épocas, patience=4) para obtener el número definitivo que va en la tabla de resultados final.

In [ ]:
best_row = sweep_df.loc[sweep_df["test_accuracy"].idxmax()]
best_nq, best_nl = int(best_row.n_qubits), int(best_row.n_layers)
print(f"Mejor combinación del barrido rápido: n_qubits={best_nq}, n_layers={best_nl} "
      f"(acc barrido={best_row.test_accuracy*100:.2f}%)")

cfg_best = Config(
    run_name=f"final_q{best_nq}_l{best_nl}_basic",
    n_qubits=best_nq,
    n_layers=best_nl,
    entangler="basic",
    epochs=15,
    early_stopping_patience=4,
)
set_seed(cfg_best.seed)
tr_loader, va_loader, te_loader = get_dataloaders(cfg_best)

best_model = HybridModel(cfg_best)
best_model, history_best, time_best = train_model(best_model, cfg_best, tr_loader, va_loader, run_name=cfg_best.run_name)
acc_best, cm_best = evaluate_model(best_model, te_loader, run_name=cfg_best.run_name)
log_experiment(cfg_best, "hybrid_final", acc_best, time_best, count_params(best_model))

print(f"\nResultado final tras barrido: {acc_best*100:.2f}% con n_qubits={best_nq}, n_layers={best_nl}")

## 14. Comparación de entanglers: `basic` vs `strong`

Aquí se aísla el efecto del **tipo de entrelazamiento** dejando fijos `n_qubits` y `n_layers` — así la única variable que cambia entre las dos corridas es cómo se conectan los qubits entre sí, no cuántos hay ni cuántas capas.

- **`BasicEntanglerLayers`**: aplica una única rotación por qubit (alrededor de un solo eje) seguida de entrelazamiento en cadena (CNOTs). Es más rápido de simular porque tiene menos parámetros por capa.
- **`StronglyEntanglingLayers`**: aplica rotaciones en los 3 ejes (X, Y, Z) por qubit antes de entrelazar. Tiene 3 veces más parámetros por capa (`weight_shape = (n_layers, n_qubits, 3)` en vez de `(n_layers, n_qubits)`), lo que le da más expresividad pero también lo hace más lento de entrenar.

**Ajusta `FIXED_QUBITS` y `FIXED_LAYERS` abajo** con la mejor combinación que haya salido de tu barrido (Sección 13) para que esta comparación use la arquitectura más prometedora que ya identificaste.

In [ ]:
# Ajusta estos dos valores según lo que haya salido mejor en el barrido de la Sección 13.
# Si aún no corriste el barrido, se usan los valores por defecto de la v1 (4 qubits, 2 capas).
FIXED_QUBITS = 8
FIXED_LAYERS = 4

entangler_results = []

for ent_type in ["basic", "strong"]:
    run_name = f"entangler_{ent_type}_q{FIXED_QUBITS}_l{FIXED_LAYERS}"
    print(f"\n=== Entrenando {run_name} ===")

    cfg_ent = Config(
        run_name=run_name,
        n_qubits=FIXED_QUBITS,
        n_layers=FIXED_LAYERS,
        entangler=ent_type,
        epochs=15,
        early_stopping_patience=4,
    )
    set_seed(cfg_ent.seed)  # mismo seed => mismo split e inicialización clásica en ambas corridas

    tr_loader, va_loader, te_loader = get_dataloaders(cfg_ent)

    model = HybridModel(cfg_ent)
    n_params = count_params(model)
    model, history, elapsed = train_model(model, cfg_ent, tr_loader, va_loader, run_name=run_name)
    acc, cm = evaluate_model(model, te_loader, run_name=run_name)
    log_experiment(cfg_ent, "hybrid_entangler_comparison", acc, elapsed, n_params)

    entangler_results.append({
        "entangler": ent_type,
        "test_accuracy": acc,
        "training_time_sec": round(elapsed, 1),
        "n_trainable_params": n_params,
        "history": history,
    })

entangler_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "history"} for r in entangler_results
])
entangler_df

### 14.1 Comparación visual: accuracy, tiempo y número de parámetros

Tres barras lado a lado para ver de un vistazo si `strong` justifica su coste extra de parámetros y tiempo con una mejora real de precisión, o si `basic` da un resultado similar más barato — esa es la pregunta que responde esta sección.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

labels = entangler_df["entangler"]

axes[0].bar(labels, entangler_df["test_accuracy"], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Accuracy en test")
axes[0].set_ylabel("Accuracy")
for i, v in enumerate(entangler_df["test_accuracy"]):
    axes[0].text(i, v, f"{v*100:.1f}%", ha="center", va="bottom")

axes[1].bar(labels, entangler_df["training_time_sec"], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Tiempo de entrenamiento")
axes[1].set_ylabel("Segundos")
for i, v in enumerate(entangler_df["training_time_sec"]):
    axes[1].text(i, v, f"{v:.0f}s", ha="center", va="bottom")

axes[2].bar(labels, entangler_df["n_trainable_params"], color=["#4C72B0", "#DD8452"])
axes[2].set_title("Parámetros entrenables")
axes[2].set_ylabel("N.º de parámetros")
for i, v in enumerate(entangler_df["n_trainable_params"]):
    axes[2].text(i, v, f"{v}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

# Resumen en texto
acc_basic = entangler_df.loc[entangler_df.entangler == "basic", "test_accuracy"].values[0]
acc_strong = entangler_df.loc[entangler_df.entangler == "strong", "test_accuracy"].values[0]
diff = (acc_strong - acc_basic) * 100
print(f"\nDiferencia de accuracy (strong - basic): {diff:+.2f} puntos porcentuales")
if diff > 1:
    print("=> 'strong' mejora de forma notable. Puede justificar el coste extra de tiempo/parámetros.")
elif diff < -1:
    print("=> 'basic' rinde mejor aquí. 'strong' no está aportando pese a su mayor coste.")
else:
    print("=> Diferencia pequeña. Con una sola semilla no es concluyente: repetir con varias semillas "
          "(ver punto 3 de 'Próximos pasos') antes de afirmar que hay diferencia real.")

### 14.2 Curvas de validación por entangler

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for r in entangler_results:
    axes[0].plot(r["history"]["epoch"], r["history"]["val_acc"], marker="o", label=r["entangler"])
    axes[1].plot(r["history"]["epoch"], r["history"]["val_loss"], marker="o", label=r["entangler"])

axes[0].set_xlabel("Época"); axes[0].set_ylabel("Val accuracy"); axes[0].set_title("Accuracy de validación")
axes[0].legend()
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Val loss"); axes[1].set_title("Pérdida de validación")
axes[1].legend()

plt.tight_layout()
plt.show()

## 15. Múltiples semillas: ¿la diferencia híbrido vs. clásico es real o es ruido?

Con tus resultados ya sabes que el clásico equivalente (94.97%) casi iguala al mejor híbrido tras todo el barrido (95.09%), en una fracción del tiempo. Pero cada número hasta ahora viene de **una sola semilla** — no puedes afirmar todavía si esa diferencia de ~0.1 puntos es real o si es solo la varianza normal de entrenar una red neuronal con inicializaciones distintas.

**Por qué solo esta comparación, y no repetir las 9 combinaciones del barrido**: cada corrida híbrida con `q8_l4` tarda ~40 minutos. Repetir las 9 configuraciones × 5 semillas sería inviable en tiempo. En cambio, la comparación **híbrido vs. clásico equivalente en la mejor arquitectura encontrada (`q8`, `l4`)** es la que responde la pregunta central del TFM — es donde vale la pena invertir el tiempo de cómputo.

**Por qué 3 semillas y no 5**: con el coste de tiempo que ya viste, 3 repeticiones por modelo (6 entrenamientos en total) es un punto razonable entre rigor estadístico y tiempo disponible. Si tienes más tiempo de máquina, añade más valores a `SEEDS_TO_TEST`.

**Aviso de tiempo**: esta celda entrena 3 semillas × 2 modelos = 6 entrenamientos completos. Con los tiempos que ya mediste (~2450s híbrido, ~415s clásico), son aproximadamente **2.5 horas** de cómputo adicional en tu máquina.

In [ ]:
# 3 semillas distintas. Cada una determina tanto la inicialización de pesos como el
# split train/val/test (recuerda que get_dataloaders usa cfg.seed para el split).
SEEDS_TO_TEST = [42, 123, 2024]

# La mejor arquitectura encontrada en el barrido (Sección 13) + basic (Sección 14)
BEST_QUBITS = 8
BEST_LAYERS = 4

multiseed_results = {"hybrid": [], "classical_equivalent": []}

for seed in SEEDS_TO_TEST:
    print(f"\n{'='*15} SEMILLA {seed} {'='*15}")

    # --- Híbrido ---
    cfg_h = Config(
        run_name=f"multiseed_hybrid_q{BEST_QUBITS}_l{BEST_LAYERS}_seed{seed}",
        n_qubits=BEST_QUBITS, n_layers=BEST_LAYERS, entangler="basic",
        epochs=15, early_stopping_patience=4, seed=seed,
    )
    set_seed(cfg_h.seed)
    tr, va, te = get_dataloaders(cfg_h)
    model_h = HybridModel(cfg_h)
    model_h, _, elapsed_h = train_model(model_h, cfg_h, tr, va, run_name=cfg_h.run_name)
    acc_h, _ = evaluate_model(model_h, te, run_name=cfg_h.run_name)
    log_experiment(cfg_h, "hybrid_multiseed", acc_h, elapsed_h, count_params(model_h))
    multiseed_results["hybrid"].append(acc_h)

    # --- Clásico equivalente (mismo seed => mismo split, misma inicialización del encoder) ---
    cfg_c = Config(
        run_name=f"multiseed_classical_q{BEST_QUBITS}_l{BEST_LAYERS}_seed{seed}",
        n_qubits=BEST_QUBITS, n_layers=BEST_LAYERS, entangler="basic",
        epochs=15, early_stopping_patience=4, seed=seed,
    )
    set_seed(cfg_c.seed)
    tr2, va2, te2 = get_dataloaders(cfg_c)
    model_c = ClassicalEquivalentModel(cfg_c)
    model_c, _, elapsed_c = train_model(model_c, cfg_c, tr2, va2, run_name=cfg_c.run_name)
    acc_c, _ = evaluate_model(model_c, te2, run_name=cfg_c.run_name)
    log_experiment(cfg_c, "classical_multiseed", acc_c, elapsed_c, count_params(model_c))
    multiseed_results["classical_equivalent"].append(acc_c)

multiseed_df = pd.DataFrame(multiseed_results, index=SEEDS_TO_TEST)
multiseed_df.index.name = "seed"
multiseed_df

### 15.1 Media ± desviación estándar, y prueba de significancia estadística

Se usa una **prueba t pareada** (`ttest_rel`), no una prueba t independiente, porque cada semilla genera el mismo split de datos para ambos modelos (híbrido y clásico se evalúan sobre el mismo test set en cada semilla) — son observaciones emparejadas, y la prueba pareada tiene más poder estadístico para detectar una diferencia real con pocas muestras.

In [ ]:
from scipy import stats

mean_h = multiseed_df["hybrid"].mean()
std_h = multiseed_df["hybrid"].std()
mean_c = multiseed_df["classical_equivalent"].mean()
std_c = multiseed_df["classical_equivalent"].std()

print(f"Híbrido (q{BEST_QUBITS}/l{BEST_LAYERS}):       {mean_h*100:.2f}% ± {std_h*100:.2f}  (n={len(SEEDS_TO_TEST)} semillas)")
print(f"Clásico equivalente:         {mean_c*100:.2f}% ± {std_c*100:.2f}  (n={len(SEEDS_TO_TEST)} semillas)")
print(f"Diferencia (clásico - híbrido): {(mean_c - mean_h)*100:+.2f} puntos porcentuales")

t_stat, p_value = stats.ttest_rel(multiseed_df["classical_equivalent"], multiseed_df["hybrid"])
print(f"\nPrueba t pareada: t={t_stat:.3f}, p={p_value:.4f}")

if p_value < 0.05:
    print("=> La diferencia ES estadísticamente significativa (p < 0.05): "
          "con esta evidencia, el modelo clásico equivalente rinde de forma distinta al híbrido, "
          "no es azar del muestreo.")
else:
    print("=> NO hay evidencia suficiente para afirmar que la diferencia es significativa con "
          "este número de semillas (p >= 0.05). Con solo 3 semillas el test tiene poca potencia "
          "estadística; si el resultado es importante para tu conclusión, considera añadir más "
          "semillas a SEEDS_TO_TEST si el tiempo de cómputo lo permite.")

### 15.2 Visualización: dispersión de resultados por semilla

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

data_to_plot = [multiseed_df["hybrid"].values, multiseed_df["classical_equivalent"].values]
bp = ax.boxplot(data_to_plot, tick_labels=["Híbrido", "Clásico equivalente"],
                patch_artist=True, widths=0.5)
for patch, color in zip(bp["boxes"], ["#4C72B0", "#DD8452"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

# Puntos individuales de cada semilla, para ver la dispersión real detrás del boxplot
for i, col in enumerate(["hybrid", "classical_equivalent"]):
    y = multiseed_df[col].values
    x = np.random.normal(i + 1, 0.04, size=len(y))
    ax.scatter(x, y, color="black", zorder=3, s=25)

ax.set_ylabel("Accuracy en test")
ax.set_title(f"Distribución de accuracy por semilla (n={len(SEEDS_TO_TEST)})")
plt.tight_layout()
plt.show()

print("Semillas usadas:", SEEDS_TO_TEST)
print("Accuracies híbrido:            ", [f'{a*100:.2f}%' for a in multiseed_df['hybrid']])
print("Accuracies clásico equivalente:", [f'{a*100:.2f}%' for a in multiseed_df['classical_equivalent']])

## 16. Experimento adicional: encoding cuántico — `AmplitudeEmbedding` vs `AngleEmbedding`

Hasta ahora el circuito codifica la información clásica con `AngleEmbedding`: cada qubit recibe **un** valor clásico como ángulo de rotación (por eso el encoder produce exactamente `n_qubits` salidas).

`AmplitudeEmbedding` es distinto: codifica un vector de **2^n_qubits** valores directamente como las amplitudes del estado cuántico. Con 8 qubits eso son 256 valores en vez de 8 — muchísima más información clásica "empaquetada" en el mismo número de qubits, pero a cambio:
- el encoder clásico previo tiene que producir 256 salidas en vez de 8 (más parámetros en esa capa),
- el vector de entrada se normaliza automáticamente a norma 1 (`normalize=True`), lo cual cambia la naturaleza de la señal que ve el circuito,
- en simuladores como `default.qubit`, preparar un `AmplitudeEmbedding` es más costoso que aplicar rotaciones, así que probablemente veas un training_time_sec mayor incluso sin cambiar `n_layers`.

Usamos la mejor combinación encontrada en el barrido (n_qubits=8, n_layers=4, entangler=basic) para que la comparación sea directa contra el resultado ya obtenido de la Sección 13.3 (95.09% con AngleEmbedding).

In [ ]:
def build_quantum_layer_amplitude(cfg: Config):
    """Variante de build_quantum_layer que usa AmplitudeEmbedding en vez de AngleEmbedding.
    Requiere un vector de entrada de tamaño 2**n_qubits (normalize=True se encarga de
    normalizarlo a norma 1 antes de cargarlo como amplitudes del estado cuántico)."""
    dev = qml.device("default.qubit", wires=cfg.n_qubits)

    if cfg.entangler == "basic":
        weight_shape = (cfg.n_layers, cfg.n_qubits)

        @qml.qnode(dev, interface="torch")
        def circuit(inputs, weights):
            qml.AmplitudeEmbedding(inputs, wires=range(cfg.n_qubits), normalize=True, pad_with=0.0)
            qml.BasicEntanglerLayers(weights, wires=range(cfg.n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(cfg.n_qubits)]

    elif cfg.entangler == "strong":
        weight_shape = (cfg.n_layers, cfg.n_qubits, 3)

        @qml.qnode(dev, interface="torch")
        def circuit(inputs, weights):
            qml.AmplitudeEmbedding(inputs, wires=range(cfg.n_qubits), normalize=True, pad_with=0.0)
            qml.StronglyEntanglingLayers(weights, wires=range(cfg.n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(cfg.n_qubits)]

    else:
        raise ValueError(f"entangler desconocido: {cfg.entangler}")

    weight_shapes = {"weights": weight_shape}
    return qml.qnn.TorchLayer(circuit, weight_shapes)


class SharedEncoderAmplitude(nn.Module):
    """Como SharedEncoder, pero produce 2**n_qubits salidas (una por amplitud)
    en vez de n_qubits (una por ángulo de rotación)."""
    def __init__(self, cfg: Config):
        super().__init__()
        n_amplitudes = 2 ** cfg.n_qubits
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, cfg.encoder_hidden_dim),
            nn.ReLU(),
            nn.Linear(cfg.encoder_hidden_dim, n_amplitudes),
        )

    def forward(self, x):
        return self.net(x)


class HybridModelAmplitude(nn.Module):
    """Igual que HybridModel pero con AmplitudeEmbedding. No multiplicamos por pi
    (eso era para convertir a ángulo); AmplitudeEmbedding normaliza internamente."""
    def __init__(self, cfg: Config):
        super().__init__()
        self.encoder = SharedEncoderAmplitude(cfg)
        self.quantum_layer = build_quantum_layer_amplitude(cfg)
        self.classifier = nn.Linear(cfg.n_qubits, 10)

    def forward(self, x):
        x = self.encoder(x)
        x = self.quantum_layer(x)
        return self.classifier(x)


### 16.1 Entrenar y comparar contra el resultado ya obtenido con `AngleEmbedding`

In [ ]:
cfg_amp = Config(
    run_name="hybrid_amplitude_q8_l4_basic",
    n_qubits=8, n_layers=4, entangler="basic",
    epochs=15, early_stopping_patience=4,
)
set_seed(cfg_amp.seed)

tr_amp, va_amp, te_amp = get_dataloaders(cfg_amp)

amplitude_model = HybridModelAmplitude(cfg_amp)
amplitude_model, history_amp, time_amp = train_model(
    amplitude_model, cfg_amp, tr_amp, va_amp, run_name=cfg_amp.run_name
)
acc_amp, cm_amp = evaluate_model(amplitude_model, te_amp, run_name=cfg_amp.run_name)
log_experiment(cfg_amp, "hybrid_amplitude", acc_amp, time_amp, count_params(amplitude_model))

# Comparación directa contra el resultado ya obtenido con AngleEmbedding (Sección 13.3 / 14)
acc_angle = 0.9509          # accuracy final de entangler_basic_q8_l4 (AngleEmbedding)
time_angle = 3180.2         # training_time_sec del mismo experimento

comparison_encoding = pd.DataFrame({
    "encoding": ["AngleEmbedding", "AmplitudeEmbedding"],
    "test_accuracy": [acc_angle, acc_amp],
    "training_time_sec": [time_angle, time_amp],
})
print(comparison_encoding)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(comparison_encoding["encoding"], comparison_encoding["test_accuracy"], color=["#4C72B0", "#DD8452"])
axes[0].set_ylabel("Test accuracy")
axes[0].set_title("Accuracy por tipo de encoding")
axes[0].set_ylim(0, 1)

axes[1].bar(comparison_encoding["encoding"], comparison_encoding["training_time_sec"], color=["#4C72B0", "#DD8452"])
axes[1].set_ylabel("Training time (s)")
axes[1].set_title("Tiempo de entrenamiento por tipo de encoding")

plt.tight_layout()
plt.show()


## Próximos pasos sugeridos para el TFM

1. **Barrido de hiperparámetros**: repetir las secciones 8-9 variando `n_qubits` (4/6/8) y `n_layers` (2/3/4) — cambiar solo el `Config` y volver a ejecutar. Con el CSV acumulado ya tienes la tabla comparativa.
2. **Comparar `basic` vs `strong` entangler** con `n_qubits` y `n_layers` fijos, para aislar el efecto del tipo de entrelazamiento.
3. **Múltiples semillas por configuración** (ej. 3-5 seeds) para poder reportar media ± desviación estándar en vez de un único número — necesario si quieres afirmar que una diferencia es estadísticamente significativa.
4. **Análisis de coste computacional**: graficar `training_time_sec` vs `test_accuracy` para todas las configuraciones — es el argumento central de cualquier TFM que evalúe si la cuántica \"merece la pena\" frente a lo clásico.
5. **Otro encoding cuántico** (`AmplitudeEmbedding` en vez de `AngleEmbedding`) como experimento adicional si da tiempo — cambia la forma en que se codifica la información clásica en el estado cuántico.
6. Cuando tengas resultados estables, pasar a estructurar el índice de la memoria (marco teórico, estado del arte, diseño experimental, resultados, conclusiones).